# TruthfulQA — Inference & Evaluation (Generalization Testing)

This notebook runs inference and evaluation on the TruthfulQA benchmark.

> **⚠️ This dataset is for generalization testing only — do not train on it.**
> TruthfulQA has only 817 samples total (~400 per class), which is too small for
> contrastive representation learning to converge reliably.
>
> **Intended workflow:** train a contrastive model on PreciseWikiQA or TriviaQA,
> then use this notebook to measure how well those representations transfer to
> TruthfulQA's distinct hallucination regime (misconception-driven vs. factual-recall).

**Dataset:** [TruthfulQA](https://huggingface.co/datasets/truthful_qa) (Yang et al., 2022) —
817 questions across 38 categories, designed so that a model mimicking human falsehoods
will answer incorrectly. Fixed HuggingFace dataset; no question-generation step.

**Steps in this notebook:**
1. Inference — generate model responses with activation logging
2. Evaluation — substring-match against `correct_answers`; write `eval_results.json`
3. Inspection — per-sample results, category breakdown, example browser
4. Cross-dataset OOD eval — load a checkpoint trained on another dataset and score TruthfulQA

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).
On `LOCAL_CPU` the inference cell will fail — run inference on the GPU host.

## 1 — Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# ── Repo root on path ──────────────────────────────────────────────────────
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# ── Model to evaluate ─────────────────────────────────────────────────────
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# ── Storage paths ──────────────────────────────────────────────────────────
base_dir          = Path("shared/truthfulqa")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "truthfulqa" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file = str(task_dir / "eval_results.json")

# ── Activation logging ─────────────────────────────────────────────────────
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# ── Inference settings ─────────────────────────────────────────────────────
MAX_TOKENS  = 128
TEMPERATURE = 0.0

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Generations file  : {generations_file}")
print(f"Eval results      : {eval_results_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 — Inference

Runs the model on all 817 TruthfulQA questions and logs intermediate layer activations.
Resume-safe: questions already present in `generation.jsonl` are skipped.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="truthfulqa",
    model=MODEL,
    N=None,                           # all 817 questions
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
)

## 3 — Evaluation

Scores each generation against TruthfulQA's `correct_answers` list (case-insensitive substring match).
Writes `eval_results.json` (ActivationParser-compatible) and `raw_eval_res.jsonl`.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="eval",
    task="truthfulqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

## 4 — Results inspection

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    eval_res = json.load(f)

print(f"Model        : {model_name}")
print(f"Total        : {eval_res['total_count']}")
print(f"Correct      : {eval_res['accurate_count']}  ({eval_res['correct_rate']:.1%})")
print(f"Hallucinated : {eval_res['hallu_count']}  ({eval_res['halu_Rate']:.1%})")

In [ ]:
# ── Per-sample view ────────────────────────────────────────────────────────
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)
print(f"Loaded {len(raw_df)} samples")
raw_df.head()

In [ ]:
# ── Hallucination rate by category ─────────────────────────────────────────
cat_stats = (
    raw_df.groupby("category")["is_hallucination"]
    .agg(halu_rate="mean", count="count")
    .sort_values("halu_rate", ascending=False)
    .reset_index()
)
cat_stats["halu_rate"] = cat_stats["halu_rate"].map("{:.1%}".format)
display(cat_stats)

In [ ]:
# ── Browse hallucinated examples ───────────────────────────────────────────
halu_df = raw_df[raw_df["is_hallucination"]].reset_index(drop=True)
print(f"Hallucinated: {len(halu_df)}")

for i, row in halu_df.head(5).iterrows():
    print(f"\n--- [{row.get('category', '')}] ---")
    print(f"Question        : {row['question']}")
    print(f"Model response  : {row['generation']}")
    print(f"Correct answers : {row['correct_answers']}")

## 5 — Cross-dataset OOD evaluation

Load a contrastive model checkpoint trained on **PreciseWikiQA or TriviaQA** and score
it against TruthfulQA activations. This tests whether hallucination representations
transfer across different hallucination types (factual-recall → misconception-driven).

Set `CHECKPOINT_PATH` to a `contrastive_last.pt` file from a prior training run.

In [ ]:
import torch
from torch.utils.data import DataLoader
from loguru import logger as _logger
_logger.remove()
_logger.add(sys.stdout, level="INFO")

from activation_logging.activation_parser import ActivationParser
from activation_research.model import ProgressiveCompressor
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator

# ── Point this at a checkpoint trained on another dataset ─────────────────
CHECKPOINT_PATH = "checkpoints/contrastive_precisewikiqa/contrastive_last.pt"  # update as needed

# ── Must match the checkpoint's architecture ───────────────────────────────
input_dim    = 4096
final_dim    = 512
target_layers = [22, 26]
relevant_layers = list(range(14, 30))
num_views    = 4
outlier_class = 1
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Verify inputs exist ────────────────────────────────────────────────────
for p, label in [(activations_path, "activations"), (eval_results_file, "eval_results"), (CHECKPOINT_PATH, "checkpoint")]:
    exists = Path(p).exists()
    print(f"{label:15s} : {'OK' if exists else 'MISSING'}  ({p})")

In [ ]:
# ── Load TruthfulQA activations ────────────────────────────────────────────
ap = ActivationParser(
    inference_json=generations_file,
    eval_json=eval_results_file,
    activations_path=activations_path,
    logger_type=LOGGER_TYPE,
    verbose=False,
)

# Use all data as the eval set (no training split needed — we're not training here)
eval_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)
train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)

print(f"TruthfulQA eval set : {len(eval_dataset)} samples")
print(ap.df["halu"].value_counts(dropna=False).rename(index={0: "non-halu", 1: "halu"}))

In [ ]:
# ── Load checkpoint ────────────────────────────────────────────────────────
model = ProgressiveCompressor(input_dim=input_dim, final_dim=final_dim, input_dropout=0.3)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval().to(device)

print(f"Loaded checkpoint from: {CHECKPOINT_PATH}")
if "train_loss" in ckpt:
    print(f"  train_loss at save : {ckpt['train_loss']:.4f}")

In [ ]:
# ── Run OOD evaluation ─────────────────────────────────────────────────────
train_ds_sliced = train_dataset.slice_layers(target_layers)
eval_ds_sliced  = eval_dataset.slice_layers(target_layers)

train_loader = DataLoader(train_ds_sliced, batch_size=64, shuffle=False)
eval_loader  = DataLoader(eval_ds_sliced,  batch_size=64, shuffle=False)

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=device,
    num_workers=4,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        "cosine",
        "mds",
        {
            "metric": "knn",
            "kwargs": {
                "k": 50,
                "metric": "euclidean",
                "calibrate_k": True,
                "k_candidates": [50, 100, 200, 500],
                "max_train_size": 200000,
                "sample_seed": 0,
            },
            "train_selection": "all",
        },
    ],
)

ood_stats = ood_eval.compute(eval_loader, model)
print("\nCross-dataset OOD results on TruthfulQA:")
for k, v in ood_stats.items():
    print(f"  {k}: {v:.4f}")